# day04

## 1. 머신러닝 애플리케이션

### 1) 모델 학습 및 저장(jupyter notebook)
- mpg_model.joblib : 학습된 모델 파일
- model_info.json : 모델 성능 지표 및 정보

### 2) Streamlit에서 모델 로드

### **자동차 연비 예측 애플리케이션**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import joblib # 모델 저장을 위한 라이브러리
import pickle # 모델 저장을 위한 라이브러리

# 데이터 준비
# seaborn의 mpg 데이터 가져오기
df = sns.load_dataset("mpg")
df.head()

In [ ]:
# 데이터 확인
df.info()

In [ ]:
# 특성(독립변수)와 타깃(종속변수) 설정
X = df[['weight']] # 무게
y = df['mpg']

# 데이터 분할(훈련, 데이터) => 7:3
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 단순 선형 회귀 모델 생성 및 학습
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
# 모델 평가
# 예측 및 평가
y_pred = model.predict(X_test)

# 결정계수
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("=== 모델 평가 결과 ===")
print(f"결정계수 : {r2 : .3f}") # 0.724 => 무게가 연비의 변화를 72%정도 설명
print(f"평균제곱오차(MSE) : {mse : .3f}")
print(f"루트평균제곱오차(RMSE) : {rmse : .3f}")
# 모델이 예측한 결과에서 약 3.9 정도 오차를 가짐

In [ ]:
# 모델 저장
# joblib 사용
joblib.dump(model, "mpg_model.joblib")
print("모델이 mpg_model.joblib로 저장되었습니다")

In [ ]:
import json
# 모델 정보 저장
model_info = {
    "r2_score" : r2, # 결정계수
    "mse" : mse, # 평균제곱오차
    "rmse" : rmse, # 루트평균오차
    "coef" : model.coef_[0], # 기울기
    "intercept" : model.intercept_, # 절편
    "feature_name" : "weight" # 독립변수(특성)
}

# json 파일로 모델 정보 저장
with open("model_info.json", "w", encoding="utf-8") as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

### Streamlit 앱 코드

아래 코드는 별도의 `.py` 파일에 저장한 뒤 `python -m streamlit run 파일명.py`로 실행합니다.

- 앱 실행 위치의 `day04` 폴더에 앞에서 저장한 `mpg_model.joblib`와 `model_info.json`을 배치합니다.
- 제공된 코드는 예측 탭까지 구현되어 있으며, 모델정보·데이터 분석 탭의 내용은 아직 없습니다.

In [ ]:
import streamlit as st
import joblib
import pandas as pd
import numpy as np
import json
import plotly.express as px
import plotly.graph_objects as go

# 페이지 설정
st.set_page_config(
    page_title = "자동차 연비 예측 앱",
    page_icon = "🚗", # 이모지 단축키 : win + .
    layout="wide"
)

# 제목 
st.title("🚗자동차 연비 예측 애플리케이션")
st.markdown("---")

# 모델 로드(캐싱)
@st.cache_resource
def load_model():
    try:
        return joblib.load("./day04/mpg_model.joblib")
    except Exception as e:
        st.error("모델 파일을 찾을 수 없습니다")
        st.error(f"{e}")
        return None

# 모델 정보 로드
def load_model_info():
    try:
        with open('./day04/model_info.json', 'r', encoding='utf-8') as f:
            return json.load(f)
    except:
        return None

# 모델 및 정보 로드
model = load_model()
model_info = load_model_info()

if model is None: # 모델 로드 실패시
    st.stop()

# 사이드 바
st.sidebar.header("⚙️ 설정")

# 모델 정보 표시
st.sidebar.subheader("📊 모델 성능")

if model_info:
    st.sidebar.metric("결정계수", f"{model_info['r2_score'] : .3f}")
    st.sidebar.metric("RMSE", f"{model_info['rmse'] : .3f}")
    st.sidebar.metric("MSE", f"{model_info['mse'] : .3f}")

# 탭 영역 나누기
tab1, tab2, tab3 = st.tabs(["🔮예측", "📈모델정보", "📊데이터 분석"])

# 탭1 : 예측
with tab1:
    st.header("연비 예측")

    col1, col2 = st.columns([2, 1])

    with col1:
        # 자동차 무게 입력폼
        st.subheader("입력 정보")
        weight = st.number_input(
            "자동차 무게 (lbs)",
            min_value=1000,
            max_value=6000,
            value=3000,
            step=100,
            help="예측할 자동차의 무게를 입력하세요"
        )

        # 예측 버튼
        if st.button("🔮 연비 예측하기", type="primary", use_container_width=True):
            # 예측 실행
            input_data = np.array([[weight]])
            predicted_mpg = model.predict(input_data)[0]
            # st.write(predicted_mpg)

            # 세션 상태에 저장
            if 'predictions' not in st.session_state:
                st.session_state.predictions = []
            st.session_state.predictions.append({
                "weight" : weight,
                "predicted_mpg" : predicted_mpg,
            })

    with col2:
        # 예측 결과 표시
        if st.session_state.get("predictions"):
            latest = st.session_state.predictions[-1]
            st.subheader("예측 결과")
            st.metric(
                label="예측 연비",
                value=f"{latest['predicted_mpg'] : .2f}"
            )
            st.caption("단위 : mpg(miles per gallon)")

            # 결과 해석
            with st.expander("📖 결과 해석"):
                st.write(f"**입력값** : {latest['weight']} lbs")
                st.write(f"**예측 연비** : {latest['predicted_mpg'] : .2f} mpg")
                st.write(f"""
                    **해석** :
                    - 무게가 {latest['weight']} lbs인 자동차의 예상 연비는 {latest['predicted_mpg'] : .2f} mpg 입니다.
                    - 이 값은 학습 데이터의 패턴을 기반으로 계산되었습니다.
                """)

                if model_info:
                    st.write(f"""
                        **오차 범위**
                        - 모델의 RMSE {model_info['rmse'] : .2f} mpg 입니다.
                        - 실제 연비는 예측값에서 평균적으로 {model_info['rmse'] : .2f} mpg 정도 차이가 날 수 있습니다
                    """)